# Ingestion V2

Langkah pertama: kosongkan vectorstore Pinecone sebelum ingest ulang.

## 1. Mengosongkan vectorstore

In [6]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from pinecone import Pinecone


def find_env_path() -> Path | None:
    candidates = [
        Path.cwd() / ".env",
        Path.cwd().parent / ".env",
        Path.cwd().parent.parent / ".env",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return None


env_path = find_env_path()

if env_path:
    load_dotenv(env_path)
    print(f"ENV loaded from: {env_path}")
else:
    load_dotenv()
    print("File .env tidak ditemukan di cwd/parent. Mencoba load dari environment aktif.")


PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

INDEX_NAME = os.getenv("INDEX_NAME_AYAT", "quran-ayat-openai")
NAMESPACE = os.getenv("PINECONE_NAMESPACE_AYAT", "ayat")

# Pilihan mode:
# - "namespace": hapus semua vector di namespace yang dipilih.
# - "default_namespace": hapus semua vector di namespace default Pinecone.
# - "all_namespaces": hapus semua vector di semua namespace pada index ini.
DELETE_MODE = "namespace"

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY belum ada di .env atau environment.")

if DELETE_MODE == "namespace" and not NAMESPACE:
    raise ValueError(
        "DELETE_MODE='namespace', tapi NAMESPACE belum diisi. "
        "Isi PINECONE_NAMESPACE_AYAT di .env, atau ubah DELETE_MODE ke 'default_namespace'."
    )

print("INDEX_NAME:", INDEX_NAME)
print("NAMESPACE:", NAMESPACE if NAMESPACE else "<default>")
print("DELETE_MODE:", DELETE_MODE)


pc = Pinecone(api_key=PINECONE_API_KEY)

if not pc.has_index(INDEX_NAME):
    raise ValueError(f"Index Pinecone tidak ditemukan: {INDEX_NAME}")

index = pc.Index(INDEX_NAME)


def describe_stats_safe():
    try:
        return index.describe_index_stats()
    except Exception as exc:
        print("Tidak bisa membaca statistik index:", exc)
        return None


def namespace_names_from_stats(stats) -> list[str]:
    if not stats:
        return []

    namespaces = getattr(stats, "namespaces", None)

    if namespaces is None and isinstance(stats, dict):
        namespaces = stats.get("namespaces")

    if not namespaces:
        return []

    return list(namespaces.keys())


stats_before = describe_stats_safe()
print("Stats sebelum delete:")
print(stats_before)

if DELETE_MODE == "namespace":
    print(f"Menghapus semua vector di namespace: {NAMESPACE}")
    index.delete(delete_all=True, namespace=NAMESPACE)

elif DELETE_MODE == "default_namespace":
    print("Menghapus semua vector di namespace default Pinecone.")
    index.delete(delete_all=True)

elif DELETE_MODE == "all_namespaces":
    namespaces = namespace_names_from_stats(stats_before)

    if not namespaces:
        print("Tidak ada namespace yang terbaca. Mencoba hapus namespace default.")
        index.delete(delete_all=True)
    else:
        for namespace in namespaces:
            print(f"Menghapus semua vector di namespace: {namespace}")
            index.delete(delete_all=True, namespace=namespace)

else:
    raise ValueError("DELETE_MODE harus 'namespace', 'default_namespace', atau 'all_namespaces'.")

print("Delete request terkirim. Menunggu Pinecone sinkron...")
time.sleep(5)

stats_after = describe_stats_safe()
print("Stats setelah delete:")
print(stats_after)

print("Vectorstore selesai dikosongkan.")

ENV loaded from: d:\Codingan Pribadi\KERJA\VIbe Coding\PINECONE\belajar-pinecone\.env
INDEX_NAME: quran-ayat-openai
NAMESPACE: ayat
DELETE_MODE: namespace
Stats sebelum delete:
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'ayat': {'vector_count': 2700}},
 'total_vector_count': 2700,
 'vector_type': 'dense'}
Menghapus semua vector di namespace: ayat
Delete request terkirim. Menunggu Pinecone sinkron...
Stats setelah delete:
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}
Vectorstore selesai dikosongkan.


## 2. Membuat JSON data vektor dari ayat_en.json

Setiap objek mewakili 1 ayat Al-Quran. Field `translation` dari sumber dipakai sebagai `content`.

In [5]:
import json
from pathlib import Path


EXPECTED_QURAN_VERSE_COUNT = 6236


def first_existing_path(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    candidate_list = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"ayat_en.json tidak ditemukan. Cek path berikut:\n{candidate_list}")


input_path = first_existing_path(
    Path("../ayat_en.json"),
    Path("ayat_en.json"),
    Path("RAW/en/ayat_en.json"),
    Path("process/RAW/en/ayat_en.json"),
)

# Output ditaruh di folder yang sama dengan ayat_en.json yang ditemukan.
output_path = input_path.with_name("ayat_en_vectorstore.json")

with open(input_path, "r", encoding="utf-8") as f:
    surahs = json.load(f)

records = []

for surah in surahs:
    surah_id = int(surah["id"])
    verses = surah.get("verses", [])

    for verse in verses:
        ayat_number = int(verse["id"])
        translation = (verse.get("translation") or "").strip()

        if not translation:
            raise ValueError(f"Translation kosong di surah {surah_id}, ayat {ayat_number}")

        metadata = {
            "id": f"surah-{surah_id}:ayat-{ayat_number}",
            "doc_type": "quran_ayat_en",
            "source": "RAW/en/ayat_en.json",
            "surah": surah_id,
            "ayat": ayat_number,
            "surah_ayat": f"{surah_id}:{ayat_number}",
            "surah_transliteration": surah.get("transliteration"),
            "surah_translation": surah.get("translation"),
            "revelation_type": surah.get("type"),
            "total_verses_in_surah": surah.get("total_verses"),
        }

        records.append({
            "content": translation,
            "metadata": {
                key: value
                for key, value in metadata.items()
                if value is not None
            }
        })

expected_from_source = sum(int(surah.get("total_verses", len(surah.get("verses", [])))) for surah in surahs)
ids = [record["metadata"]["id"] for record in records]

if len(records) != expected_from_source:
    raise ValueError(
        f"Jumlah records ({len(records)}) tidak sama dengan total_verses di sumber ({expected_from_source})."
    )

if len(records) != EXPECTED_QURAN_VERSE_COUNT:
    raise ValueError(
        f"Jumlah records ({len(records)}) tidak sama dengan jumlah ayat Al-Quran ({EXPECTED_QURAN_VERSE_COUNT})."
    )

if len(ids) != len(set(ids)):
    raise ValueError("Ada ID duplikat. Cek kombinasi surah dan ayat di ayat_en.json.")

output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print("Input:", input_path)
print("Output:", output_path)
print("Jumlah objek:", len(records))
print("Contoh objek pertama:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))
print("Contoh objek terakhir:")
print(json.dumps(records[-1], ensure_ascii=False, indent=2))

Input: D:\Codingan Pribadi\KERJA\VIbe Coding\qurandataset\quranqadataset\ayat_en.json
Output: D:\Codingan Pribadi\KERJA\VIbe Coding\qurandataset\quranqadataset\ayat_en_vectorstore.json
Jumlah objek: 6236
Contoh objek pertama:
{
  "content": "In the name of Allah, the Entirely Merciful, the Especially Merciful",
  "metadata": {
    "id": "surah-1:ayat-1",
    "doc_type": "quran_ayat_en",
    "source": "RAW/en/ayat_en.json",
    "surah": 1,
    "ayat": 1,
    "surah_ayat": "1:1",
    "surah_transliteration": "Al-Fatihah",
    "surah_translation": "The Opener",
    "revelation_type": "meccan",
    "total_verses_in_surah": 7
  }
}
Contoh objek terakhir:
{
  "content": "From among the jinn and mankind",
  "metadata": {
    "id": "surah-114:ayat-6",
    "doc_type": "quran_ayat_en",
    "source": "RAW/en/ayat_en.json",
    "surah": 114,
    "ayat": 6,
    "surah_ayat": "114:6",
    "surah_transliteration": "An-Nas",
    "surah_translation": "Mankind",
    "revelation_type": "meccan",
    "tot

## 3. Ingest ayat_en_vectorstore.json ke Pinecone

Embedding dibuat hanya dari field `content`. Metadata disimpan untuk filtering/identitas, tetapi `content` tidak ikut dimasukkan ke metadata Pinecone.

In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from tqdm.auto import tqdm

from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone, ServerlessSpec


def find_env_path() -> Path | None:
    candidates = [
        Path.cwd() / ".env",
        Path.cwd().parent / ".env",
        Path.cwd().parent.parent / ".env",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    return None


def first_existing_path(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    candidate_list = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"File input tidak ditemukan. Cek path berikut:\n{candidate_list}")


env_path = find_env_path()

if env_path:
    load_dotenv(env_path)
    print(f"ENV loaded from: {env_path}")
else:
    load_dotenv()
    print("File .env tidak ditemukan di cwd/parent. Mencoba load dari environment aktif.")


PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

INDEX_NAME = os.getenv("INDEX_NAME_AYAT", "quran-ayat-openai")
NAMESPACE = os.getenv("PINECONE_NAMESPACE_AYAT", "ayat")

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMENSION = 1536
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")

EMBEDDING_BATCH_SIZE = 100
UPSERT_BATCH_SIZE = 100

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY belum ada di .env atau environment.")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY belum ada di .env atau environment.")

print("INDEX_NAME:", INDEX_NAME)
print("NAMESPACE:", NAMESPACE)
print("EMBEDDING_MODEL:", EMBEDDING_MODEL)


# =========================
# 1. Load JSON vectorstore
# =========================

data_path = first_existing_path(
    Path("../ayat_en_vectorstore.json"),
    Path("ayat_en_vectorstore.json"),
    Path("process/ayat_en_vectorstore.json"),
)

with open(data_path, "r", encoding="utf-8") as f:
    records = json.load(f)

records = [
    record
    for record in records
    if (record.get("content") or "").strip()
]

print("Input:", data_path)
print("Jumlah records:", len(records))
print("Contoh record:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))


# =========================
# 2. Clean metadata
# =========================

def clean_metadata_value(value: Any):
    if value is None:
        return None

    if isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, list):
        return [str(item) for item in value if item is not None]

    return str(value)


def clean_metadata(metadata: dict) -> dict:
    cleaned = {}

    for key, value in (metadata or {}).items():
        key = str(key)

        if key.startswith("$"):
            key = key.replace("$", "_", 1)

        cleaned_value = clean_metadata_value(value)

        if cleaned_value is not None:
            cleaned[key] = cleaned_value

    return cleaned


cleaned_records = []

for i, record in enumerate(records):
    content = record["content"].strip()
    metadata = clean_metadata(record.get("metadata", {}))

    doc_id = metadata.get("id") or f"ayat-en-{i}"
    metadata["id"] = str(doc_id)

    cleaned_records.append({
        "id": str(doc_id),
        "content": content,
        "metadata": metadata,
    })

records = cleaned_records
ids = [record["id"] for record in records]

print("Jumlah ID:", len(ids))
print("Jumlah unique ID:", len(set(ids)))

if len(ids) != len(set(ids)):
    raise ValueError("Ada ID duplikat. Cek metadata id di ayat_en_vectorstore.json.")


# =========================
# 3. Setup Pinecone index
# =========================

pc = Pinecone(api_key=PINECONE_API_KEY)

if not pc.has_index(INDEX_NAME):
    print(f"Membuat index baru: {INDEX_NAME}")

    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,
            region=PINECONE_REGION,
        ),
    )
else:
    print(f"Index sudah ada: {INDEX_NAME}")

while True:
    desc = pc.describe_index(INDEX_NAME)

    try:
        is_ready = desc.status["ready"]
    except Exception:
        is_ready = desc.status.ready

    if is_ready:
        break

    time.sleep(2)

desc = pc.describe_index(INDEX_NAME)
index_dimension = getattr(desc, "dimension", None)

if index_dimension is not None and index_dimension != EMBEDDING_DIMENSION:
    raise ValueError(
        f"Dimension index Pinecone = {index_dimension}, "
        f"tapi embedding model {EMBEDDING_MODEL} butuh {EMBEDDING_DIMENSION}. "
        "Buat index baru atau pakai model embedding yang sesuai."
    )

index = pc.Index(INDEX_NAME)


# =========================
# 4. Simpan content lookup lokal
# =========================
# Content tidak dikirim ke metadata Pinecone. Lookup ini dipakai untuk menampilkan isi ayat dari ID hasil search.

content_lookup = {
    record["id"]: {
        "content": record["content"],
        "metadata": record["metadata"],
    }
    for record in records
}

lookup_path = data_path.parent / "ayat_en_content_lookup.json"

with open(lookup_path, "w", encoding="utf-8") as f:
    json.dump(content_lookup, f, ensure_ascii=False, indent=2)

print("Content lookup tersimpan di:", lookup_path)


# =========================
# 5. Embed content saja, lalu upsert
# =========================

embedding = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=OPENAI_API_KEY,
)

for start in tqdm(range(0, len(records), EMBEDDING_BATCH_SIZE)):
    end = start + EMBEDDING_BATCH_SIZE
    batch_records = records[start:end]

    # Hanya field content yang diubah menjadi embedding/vector.
    batch_texts = [record["content"] for record in batch_records]
    batch_vectors = embedding.embed_documents(batch_texts)

    pinecone_vectors = []

    for record, vector in zip(batch_records, batch_vectors):
        pinecone_vectors.append({
            "id": record["id"],
            "values": vector,
            "metadata": record["metadata"],
        })

    for upsert_start in range(0, len(pinecone_vectors), UPSERT_BATCH_SIZE):
        upsert_end = upsert_start + UPSERT_BATCH_SIZE
        index.upsert(
            vectors=pinecone_vectors[upsert_start:upsert_end],
            namespace=NAMESPACE,
        )

print("Ingestion selesai.")
print("Yang di-embed menjadi vector: content saja.")
print("Content tidak disimpan di metadata Pinecone.")

## 4. Retrieval dari vectorstore

Query diubah menjadi embedding dengan model yang sama. Hasil Pinecone berisi ID, score, dan metadata; isi `content` diambil dari `ayat_en_content_lookup.json`.

In [9]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone


def find_env_path() -> Path | None:
    candidates = [
        Path.cwd() / ".env",
        Path.cwd().parent / ".env",
        Path.cwd().parent.parent / ".env",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    return None


def first_existing_path(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    candidate_list = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"File tidak ditemukan. Cek path berikut:\n{candidate_list}")


env_path = find_env_path()

if env_path:
    load_dotenv(env_path)
    print(f"ENV loaded from: {env_path}")
else:
    load_dotenv()
    print("File .env tidak ditemukan di cwd/parent. Mencoba load dari environment aktif.")


PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

INDEX_NAME = os.getenv("INDEX_NAME_AYAT", "quran-ayat-openai")
NAMESPACE = os.getenv("PINECONE_NAMESPACE_AYAT", "ayat")
EMBEDDING_MODEL = "text-embedding-3-small"

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY belum ada di .env atau environment.")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY belum ada di .env atau environment.")


try:
    lookup_path = first_existing_path(
        Path("../ayat_en_content_lookup.json"),
        Path("ayat_en_content_lookup.json"),
        Path("process/ayat_en_content_lookup.json"),
    )

    with open(lookup_path, "r", encoding="utf-8") as f:
        content_lookup = json.load(f)
except FileNotFoundError:
    data_path = first_existing_path(
        Path("../ayat_en_vectorstore.json"),
        Path("ayat_en_vectorstore.json"),
        Path("process/ayat_en_vectorstore.json"),
    )

    with open(data_path, "r", encoding="utf-8") as f:
        vectorstore_records = json.load(f)

    content_lookup = {
        record["metadata"]["id"]: {
            "content": record["content"],
            "metadata": record.get("metadata", {}),
        }
        for record in vectorstore_records
        if record.get("metadata", {}).get("id") and (record.get("content") or "").strip()
    }

    lookup_path = data_path.with_name("ayat_en_content_lookup.json")
    with open(lookup_path, "w", encoding="utf-8") as f:
        json.dump(content_lookup, f, ensure_ascii=False, indent=2)

    print("Content lookup belum ada, dibuat dari:", data_path)

print("INDEX_NAME:", INDEX_NAME)
print("NAMESPACE:", NAMESPACE)
print("Content lookup:", lookup_path)


embedding = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=OPENAI_API_KEY,
)

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)


# Ganti query ini sesuai kebutuhan.
query = "worship Allah alone"
top_k = 5

# Query juga di-embed dari teks biasa, agar cocok dengan vector content saat ingestion.
query_vector = embedding.embed_query(query)

response = index.query(
    vector=query_vector,
    namespace=NAMESPACE,
    top_k=top_k,
    include_values=False,
    include_metadata=True,
)

matches = response.get("matches", [])

print("QUERY:", query)
print("HASIL RETRIEVAL:")

if not matches:
    print("Tidak ada hasil.")
else:
    for rank, match in enumerate(matches, start=1):
        doc_id = match.get("id")
        score = match.get("score")
        metadata = match.get("metadata", {}) or {}
        lookup_item = content_lookup.get(doc_id, {})
        content = lookup_item.get("content", "")

        print("=" * 80)
        print("Rank:", rank)
        print("ID:", doc_id)
        print("Score:", score)
        print("Surah/Ayat:", metadata.get("surah_ayat"))
        print("Surah:", metadata.get("surah_transliteration"))
        print("Content:", content)
        print("Metadata:", metadata)

ENV loaded from: D:\Codingan Pribadi\KERJA\VIbe Coding\qurandataset\quranqadataset\.env
Content lookup belum ada, dibuat dari: D:\Codingan Pribadi\KERJA\VIbe Coding\qurandataset\quranqadataset\ayat_en_vectorstore.json
INDEX_NAME: quran-ayat-openai
NAMESPACE: ayat
Content lookup: D:\Codingan Pribadi\KERJA\VIbe Coding\qurandataset\quranqadataset\ayat_en_content_lookup.json


c:\Users\ASUS\.conda\envs\dataquran\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


QUERY: worship Allah alone
HASIL RETRIEVAL:
Rank: 1
ID: surah-39:ayat-14
Score: 0.740643501
Surah/Ayat: 39:14
Surah: Az-Zumar
Content: Say, "Allah [alone] do I worship, sincere to Him in my religion
Metadata: {'ayat': 14.0, 'doc_type': 'quran_ayat_en', 'id': 'surah-39:ayat-14', 'revelation_type': 'meccan', 'source': 'RAW/en/ayat_en.json', 'surah': 39.0, 'surah_ayat': '39:14', 'surah_translation': 'The Troops', 'surah_transliteration': 'Az-Zumar', 'total_verses_in_surah': 75.0}
Rank: 2
ID: surah-39:ayat-66
Score: 0.68764168
Surah/Ayat: 39:66
Surah: Az-Zumar
Content: Rather, worship [only] Allah and be among the grateful
Metadata: {'ayat': 66.0, 'doc_type': 'quran_ayat_en', 'id': 'surah-39:ayat-66', 'revelation_type': 'meccan', 'source': 'RAW/en/ayat_en.json', 'surah': 39.0, 'surah_ayat': '39:66', 'surah_translation': 'The Troops', 'surah_transliteration': 'Az-Zumar', 'total_verses_in_surah': 75.0}
Rank: 3
ID: surah-53:ayat-62
Score: 0.68253535
Surah/Ayat: 53:62
Surah: An-Najm
Content: S

## 5. Membuat JSON data vektor dari ayat_id.json

Bagian ini membuat file vectorstore lokal untuk terjemahan Bahasa Indonesia. Field `content` berisi terjemahan ayat Bahasa Indonesia yang akan di-embed.

In [12]:
import json
from pathlib import Path


EXPECTED_QURAN_VERSE_COUNT = 6236


def first_existing_path(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    candidate_list = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"ayat_id.json tidak ditemukan. Cek path berikut:\n{candidate_list}")


input_path = first_existing_path(
    Path("ayat_id.json"),
    Path("silab/ayat_id.json"),
    Path("../ayat_id.json"),
    Path("RAW/id/ayat_id.json"),
    Path("process/RAW/id/ayat_id.json"),
)

# Output ditaruh di folder yang sama dengan ayat_id.json yang ditemukan.
output_path = input_path.with_name("ayat_id_vectorstore.json")

with open(input_path, "r", encoding="utf-8") as f:
    surahs = json.load(f)

records = []

for surah in surahs:
    surah_id = int(surah["id"])
    verses = surah.get("verses", [])

    for verse in verses:
        ayat_number = int(verse["id"])
        translation = (verse.get("translation") or "").strip()
        arabic_text = (verse.get("text") or "").strip()

        if not translation:
            raise ValueError(f"Translation kosong di surah {surah_id}, ayat {ayat_number}")

        metadata = {
            "id": f"surah-{surah_id}:ayat-{ayat_number}",
            "doc_type": "quran_ayat_id",
            "source": "ayat_id.json",
            "language": "id",
            "surah": surah_id,
            "ayat": ayat_number,
            "surah_ayat": f"{surah_id}:{ayat_number}",
            "surah_name_arabic": surah.get("name"),
            "surah_transliteration": surah.get("transliteration"),
            "surah_translation": surah.get("translation"),
            "revelation_type": surah.get("type"),
            "total_verses_in_surah": surah.get("total_verses"),
            "arabic_text": arabic_text,
        }

        records.append({
            "content": translation,
            "metadata": {
                key: value
                for key, value in metadata.items()
                if value is not None and value != ""
            }
        })

expected_from_source = sum(int(surah.get("total_verses", len(surah.get("verses", [])))) for surah in surahs)
ids = [record["metadata"]["id"] for record in records]

if len(records) != expected_from_source:
    raise ValueError(
        f"Jumlah records ({len(records)}) tidak sama dengan total_verses di sumber ({expected_from_source})."
    )

if len(records) != EXPECTED_QURAN_VERSE_COUNT:
    raise ValueError(
        f"Jumlah records ({len(records)}) tidak sama dengan jumlah ayat Al-Quran ({EXPECTED_QURAN_VERSE_COUNT})."
    )

if len(ids) != len(set(ids)):
    raise ValueError("Ada ID duplikat. Cek kombinasi surah dan ayat di ayat_id.json.")

output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print("Input:", input_path)
print("Output:", output_path)
print("Jumlah objek:", len(records))
print("Contoh objek pertama:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))
print("Contoh objek terakhir:")
print(json.dumps(records[-1], ensure_ascii=False, indent=2))


Input: D:\Codingan Pribadi\KERJA\VIbe Coding\qurandataset\quranqadataset\silab\ayat_id.json
Output: D:\Codingan Pribadi\KERJA\VIbe Coding\qurandataset\quranqadataset\silab\ayat_id_vectorstore.json
Jumlah objek: 6236
Contoh objek pertama:
{
  "content": "Dengan nama Allah Yang Maha Pengasih, Maha Penyayang",
  "metadata": {
    "id": "surah-1:ayat-1",
    "doc_type": "quran_ayat_id",
    "source": "ayat_id.json",
    "language": "id",
    "surah": 1,
    "ayat": 1,
    "surah_ayat": "1:1",
    "surah_name_arabic": "الفاتحة",
    "surah_transliteration": "Al-Fatihah",
    "surah_translation": "Pembukaan",
    "revelation_type": "meccan",
    "total_verses_in_surah": 7,
    "arabic_text": "بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ"
  }
}
Contoh objek terakhir:
{
  "content": "dari (golongan) jin dan manusia",
  "metadata": {
    "id": "surah-114:ayat-6",
    "doc_type": "quran_ayat_id",
    "source": "ayat_id.json",
    "language": "id",
    "surah": 114,
    "ayat": 6,
    "surah_ayat": "11

## 6. Ingest ayat_id_vectorstore.json ke Pinecone

Bagian ini membuat/memakai vectorstore Pinecone baru untuk Bahasa Indonesia. Default index: `quran-ayat-id-openai`, namespace: `ayat_id`.

In [13]:
import json
import os
import time
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from tqdm.auto import tqdm

from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone, ServerlessSpec


def find_env_path() -> Path | None:
    candidates = [
        Path.cwd() / ".env",
        Path.cwd().parent / ".env",
        Path.cwd().parent.parent / ".env",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    return None


def first_existing_path(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    candidate_list = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"File input tidak ditemukan. Cek path berikut:\n{candidate_list}")


env_path = find_env_path()

if env_path:
    load_dotenv(env_path)
    print(f"ENV loaded from: {env_path}")
else:
    load_dotenv()
    print("File .env tidak ditemukan di cwd/parent. Mencoba load dari environment aktif.")


PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

INDEX_NAME_ID = os.getenv("INDEX_NAME_AYAT_ID", "quran-ayat-id-openai")
NAMESPACE_ID = os.getenv("PINECONE_NAMESPACE_AYAT_ID", "ayat_id")

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMENSION = 1536
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")

EMBEDDING_BATCH_SIZE = 100
UPSERT_BATCH_SIZE = 100

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY belum ada di .env atau environment.")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY belum ada di .env atau environment.")

print("INDEX_NAME_ID:", INDEX_NAME_ID)
print("NAMESPACE_ID:", NAMESPACE_ID)
print("EMBEDDING_MODEL:", EMBEDDING_MODEL)


# =========================
# 1. Load JSON vectorstore Bahasa Indonesia
# =========================

data_path = first_existing_path(
    Path("ayat_id_vectorstore.json"),
    Path("silab/ayat_id_vectorstore.json"),
    Path("../ayat_id_vectorstore.json"),
    Path("process/ayat_id_vectorstore.json"),
)

with open(data_path, "r", encoding="utf-8") as f:
    records = json.load(f)

records = [
    record
    for record in records
    if (record.get("content") or "").strip()
]

print("Input:", data_path)
print("Jumlah records:", len(records))
print("Contoh record:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))


# =========================
# 2. Clean metadata
# =========================

def clean_metadata_value(value: Any):
    if value is None:
        return None

    if isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, list):
        return [str(item) for item in value if item is not None]

    return str(value)


def clean_metadata(metadata: dict) -> dict:
    cleaned = {}

    for key, value in (metadata or {}).items():
        key = str(key)

        if key.startswith("$"):
            key = key.replace("$", "_", 1)

        cleaned_value = clean_metadata_value(value)

        if cleaned_value is not None:
            cleaned[key] = cleaned_value

    return cleaned


cleaned_records = []

for i, record in enumerate(records):
    content = record["content"].strip()
    metadata = clean_metadata(record.get("metadata", {}))

    doc_id = metadata.get("id") or f"ayat-id-{i}"
    metadata["id"] = str(doc_id)

    cleaned_records.append({
        "id": str(doc_id),
        "content": content,
        "metadata": metadata,
    })

records = cleaned_records
ids = [record["id"] for record in records]

print("Jumlah ID:", len(ids))
print("Jumlah unique ID:", len(set(ids)))

if len(ids) != len(set(ids)):
    raise ValueError("Ada ID duplikat. Cek metadata id di ayat_id_vectorstore.json.")


# =========================
# 3. Setup Pinecone index baru Bahasa Indonesia
# =========================

pc = Pinecone(api_key=PINECONE_API_KEY)

if not pc.has_index(INDEX_NAME_ID):
    print(f"Membuat index baru: {INDEX_NAME_ID}")

    pc.create_index(
        name=INDEX_NAME_ID,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,
            region=PINECONE_REGION,
        ),
    )
else:
    print(f"Index sudah ada: {INDEX_NAME_ID}")

while True:
    desc = pc.describe_index(INDEX_NAME_ID)

    try:
        is_ready = desc.status["ready"]
    except Exception:
        is_ready = desc.status.ready

    if is_ready:
        break

    time.sleep(2)

desc = pc.describe_index(INDEX_NAME_ID)
index_dimension = getattr(desc, "dimension", None)

if index_dimension is not None and index_dimension != EMBEDDING_DIMENSION:
    raise ValueError(
        f"Dimension index Pinecone = {index_dimension}, "
        f"tapi embedding model {EMBEDDING_MODEL} butuh {EMBEDDING_DIMENSION}. "
        "Buat index baru atau pakai model embedding yang sesuai."
    )

index = pc.Index(INDEX_NAME_ID)


# =========================
# 4. Simpan content lookup lokal Bahasa Indonesia
# =========================
# Content tidak dikirim ke metadata Pinecone. Lookup ini dipakai untuk menampilkan isi ayat dari ID hasil search.

content_lookup = {
    record["id"]: {
        "content": record["content"],
        "metadata": record["metadata"],
    }
    for record in records
}

lookup_path = data_path.with_name("ayat_id_content_lookup.json")

with open(lookup_path, "w", encoding="utf-8") as f:
    json.dump(content_lookup, f, ensure_ascii=False, indent=2)

print("Content lookup Bahasa Indonesia tersimpan di:", lookup_path)


# =========================
# 5. Embed content saja, lalu upsert ke vectorstore baru
# =========================

embedding = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=OPENAI_API_KEY,
)

for start in tqdm(range(0, len(records), EMBEDDING_BATCH_SIZE)):
    end = start + EMBEDDING_BATCH_SIZE
    batch_records = records[start:end]

    # Hanya field content Bahasa Indonesia yang diubah menjadi embedding/vector.
    batch_texts = [record["content"] for record in batch_records]
    batch_vectors = embedding.embed_documents(batch_texts)

    pinecone_vectors = []

    for record, vector in zip(batch_records, batch_vectors):
        pinecone_vectors.append({
            "id": record["id"],
            "values": vector,
            "metadata": record["metadata"],
        })

    for upsert_start in range(0, len(pinecone_vectors), UPSERT_BATCH_SIZE):
        upsert_end = upsert_start + UPSERT_BATCH_SIZE
        index.upsert(
            vectors=pinecone_vectors[upsert_start:upsert_end],
            namespace=NAMESPACE_ID,
        )

print("Ingestion Bahasa Indonesia selesai.")
print("Yang di-embed menjadi vector: content Bahasa Indonesia saja.")
print("Content tidak disimpan di metadata Pinecone.")
print("Index:", INDEX_NAME_ID)
print("Namespace:", NAMESPACE_ID)


ENV loaded from: D:\Codingan Pribadi\KERJA\VIbe Coding\qurandataset\quranqadataset\.env
INDEX_NAME_ID: quran-ayat-id-openai
NAMESPACE_ID: ayat_id
EMBEDDING_MODEL: text-embedding-3-small
Input: D:\Codingan Pribadi\KERJA\VIbe Coding\qurandataset\quranqadataset\silab\ayat_id_vectorstore.json
Jumlah records: 6236
Contoh record:
{
  "content": "Dengan nama Allah Yang Maha Pengasih, Maha Penyayang",
  "metadata": {
    "id": "surah-1:ayat-1",
    "doc_type": "quran_ayat_id",
    "source": "ayat_id.json",
    "language": "id",
    "surah": 1,
    "ayat": 1,
    "surah_ayat": "1:1",
    "surah_name_arabic": "الفاتحة",
    "surah_transliteration": "Al-Fatihah",
    "surah_translation": "Pembukaan",
    "revelation_type": "meccan",
    "total_verses_in_surah": 7,
    "arabic_text": "بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ"
  }
}
Jumlah ID: 6236
Jumlah unique ID: 6236
Index sudah ada: quran-ayat-id-openai
Content lookup Bahasa Indonesia tersimpan di: D:\Codingan Pribadi\KERJA\VIbe Coding\qurandata

100%|██████████| 63/63 [13:33<00:00, 12.91s/it]

Ingestion Bahasa Indonesia selesai.
Yang di-embed menjadi vector: content Bahasa Indonesia saja.
Content tidak disimpan di metadata Pinecone.
Index: quran-ayat-id-openai
Namespace: ayat_id


## 7. Retrieval dari vectorstore Bahasa Indonesia

Bagian ini mengambil hasil dari vectorstore Bahasa Indonesia yang dibuat pada langkah sebelumnya.

In [2]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone


def find_env_path() -> Path | None:
    candidates = [
        Path.cwd() / ".env",
        Path.cwd().parent / ".env",
        Path.cwd().parent.parent / ".env",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    return None


def first_existing_path(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    candidate_list = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"File tidak ditemukan. Cek path berikut:\n{candidate_list}")


env_path = find_env_path()

if env_path:
    load_dotenv(env_path)
    print(f"ENV loaded from: {env_path}")
else:
    load_dotenv()
    print("File .env tidak ditemukan di cwd/parent. Mencoba load dari environment aktif.")


PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

INDEX_NAME_ID = os.getenv("INDEX_NAME_AYAT_ID", "quran-ayat-id-openai")
NAMESPACE_ID = os.getenv("PINECONE_NAMESPACE_AYAT_ID", "ayat_id")
EMBEDDING_MODEL = "text-embedding-3-small"

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY belum ada di .env atau environment.")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY belum ada di .env atau environment.")


try:
    lookup_path = first_existing_path(
        Path("ayat_id_content_lookup.json"),
        Path("silab/ayat_id_content_lookup.json"),
        Path("../ayat_id_content_lookup.json"),
        Path("process/ayat_id_content_lookup.json"),
    )

    with open(lookup_path, "r", encoding="utf-8") as f:
        content_lookup = json.load(f)
except FileNotFoundError:
    data_path = first_existing_path(
        Path("ayat_id_vectorstore.json"),
        Path("silab/ayat_id_vectorstore.json"),
        Path("../ayat_id_vectorstore.json"),
        Path("process/ayat_id_vectorstore.json"),
    )

    with open(data_path, "r", encoding="utf-8") as f:
        vectorstore_records = json.load(f)

    content_lookup = {
        record["metadata"]["id"]: {
            "content": record["content"],
            "metadata": record.get("metadata", {}),
        }
        for record in vectorstore_records
        if record.get("metadata", {}).get("id") and (record.get("content") or "").strip()
    }

    lookup_path = data_path.with_name("ayat_id_content_lookup.json")
    with open(lookup_path, "w", encoding="utf-8") as f:
        json.dump(content_lookup, f, ensure_ascii=False, indent=2)

    print("Content lookup Bahasa Indonesia belum ada, dibuat dari:", data_path)

print("INDEX_NAME_ID:", INDEX_NAME_ID)
print("NAMESPACE_ID:", NAMESPACE_ID)
print("Content lookup:", lookup_path)


embedding = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=OPENAI_API_KEY,
)

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME_ID)


# Ganti query ini sesuai kebutuhan.
query = "menyembah Allah saja"
top_k = 5

# Query juga di-embed dari teks Bahasa Indonesia, agar cocok dengan vector content Bahasa Indonesia.
query_vector = embedding.embed_query(query)

response = index.query(
    vector=query_vector,
    namespace=NAMESPACE_ID,
    top_k=top_k,
    include_values=False,
    include_metadata=True,
)

matches = response.get("matches", [])

print("QUERY:", query)
print("HASIL RETRIEVAL BAHASA INDONESIA:")

if not matches:
    print("Tidak ada hasil.")
else:
    for rank, match in enumerate(matches, start=1):
        doc_id = match.get("id")
        score = match.get("score")
        metadata = match.get("metadata", {}) or {}
        lookup_item = content_lookup.get(doc_id, {})
        content = lookup_item.get("content", "")

        print("=" * 80)
        print("Rank:", rank)
        print("ID:", doc_id)
        print("Score:", score)
        print("Surah/Ayat:", metadata.get("surah_ayat"))
        print("Surah:", metadata.get("surah_transliteration"))
        print("Content ID:", content)
        print("Metadata:", metadata)


ENV loaded from: D:\Codingan Pribadi\KERJA\VIbe Coding\qurandataset\quranqadataset\.env
INDEX_NAME_ID: quran-ayat-id-openai
NAMESPACE_ID: ayat_id
Content lookup: D:\Codingan Pribadi\KERJA\VIbe Coding\qurandataset\quranqadataset\silab\ayat_id_content_lookup.json


c:\Users\ASUS\.conda\envs\dataquran\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


QUERY: menyembah Allah saja
HASIL RETRIEVAL BAHASA INDONESIA:
Rank: 1
ID: surah-112:ayat-2
Score: 0.627985775
Surah/Ayat: 112:2
Surah: Al-Ikhlas
Content ID: Allah tempat meminta segala sesuatu
Metadata: {'arabic_text': 'ٱللَّهُ ٱلصَّمَدُ', 'ayat': 2.0, 'doc_type': 'quran_ayat_id', 'id': 'surah-112:ayat-2', 'language': 'id', 'revelation_type': 'meccan', 'source': 'ayat_id.json', 'surah': 112.0, 'surah_ayat': '112:2', 'surah_name_arabic': 'الإخلاص', 'surah_translation': 'Ikhlash', 'surah_transliteration': 'Al-Ikhlas', 'total_verses_in_surah': 4.0}
Rank: 2
ID: surah-53:ayat-42
Score: 0.622477591
Surah/Ayat: 53:42
Surah: An-Najm
Content ID: dan sesungguhnya kepada Tuhanmulah kesudahannya (segala sesuatu)
Metadata: {'arabic_text': 'وَأَنَّ إِلَىٰ رَبِّكَ ٱلۡمُنتَهَىٰ', 'ayat': 42.0, 'doc_type': 'quran_ayat_id', 'id': 'surah-53:ayat-42', 'language': 'id', 'revelation_type': 'meccan', 'source': 'ayat_id.json', 'surah': 53.0, 'surah_ayat': '53:42', 'surah_name_arabic': 'النجم', 'surah_translat